<a href="https://colab.research.google.com/github/jazoza/cultural-data-analysis/blob/main/04_CDA_HH_midterm_synthesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cultural Data Analysis

Synthesis
Week 01 - Week 04

In [ ]:
# import necessary libraries
import os, re, csv
import numpy as np
import pandas as pd

# Heritage Houses Websites

## Loading the datasets: heritage homes websites

The dataset is stored in a shared google drive:
https://drive.google.com/drive/folders/11Shm0edDOiWrOe56fzJQRZi-v_BPSW8E?usp=drive_link

Add it to your drive.

To access it, load your gdrive in 'Files' (see left pane of the notebook in google colab) and navigate to the shared folder. You may need to click on 'refresh' to make it appear on the list.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Load the github repository files where this notebook is stored to the available files. This will make it easier to import stopwords, url lists and other additional data we need.

In [ ]:
!git clone https://github.com/jazoza/cultural-data-analysis

## Import all datasets (4 countries)

You will have all datasets available for analysis and comparison, mapped in the following way:

> df0 - Dutch dataset

> df1 - UK dataset

> df2 - German dataset

> df3 - French dataset

In [ ]:
# Country code: change here between 'NL' and 'UK'
cc_list = ['NL', 'UK', 'DE', 'FR']

In [ ]:
gdrive_path = '/content/gdrive/MyDrive/CDA/'

In [ ]:
# Import json data from Aipfy scraping into 4 separate dataframes
df_NL=pd.read_json(gdrive_path+cc_list[0]+'_dataset_website-content-crawler.json') # NL
# select only two columns for analysis: url and text
df_NL=df_NL[['url','text']]

df_UK=pd.read_json(gdrive_path+cc_list[1]+'_dataset_website-content-crawler.json') # UK
# select only two columns for analysis: url and text
df_UK=df_UK[['url','text']]

df_DE=pd.read_json(gdrive_path+cc_list[2]+'_dataset_website-content-crawler.json') # DE
# select only two columns for analysis: url and text
df_DE=df_DE[['url','text']]

df_FR=pd.read_json(gdrive_path+cc_list[3]+'_dataset_website-content-crawler.json') # FR
# select only two columns for analysis: url and text
df_FR=df_FR[['url','text']]

df_FR.head()

In [ ]:
len(df_UK)

Join all pages from a domain to an entry in the analysis. To do this, add a new column which will contain only the main domain name.

In [ ]:
import re

# function to extract the main domain from the url in the dataset
def extract_main_domain(url):
    if not isinstance(str(url), str):
        print('NOT VALID',url)
        return None
    match = re.findall(r'(?:\w+\.)*\w+\.\w*', str(url))
    return match[0].removeprefix('www.') if match else None

In [ ]:
# Add a new column 'domain' and fill it by applying the extract_main_domain function to the 'url' column

# first, create a mapping of dataframes which could be addressed in a loop
df_dict = {'0':df_NL, '1':df_UK, '2':df_DE, '3':df_FR}

# then, loop through the df_dict to update each dataframe
for k, v in df_dict.items():
  cc_column = cc_list[int(k[-1])]+' domains'
  cc = cc_list[int(k[-1])]
  # print(cc_column, cc)
  urls = pd.read_csv(f'{gdrive_path}url_lists/{cc_list[int(k[-1])]}_urls.csv')[cc_column].values.tolist()
  domains = {extract_main_domain(url) for url in urls if extract_main_domain(url) is not None}
  matching_links = [link for link in v.url if extract_main_domain(link) in domains]
  # update the dataframe
  v['domain'] = v['url'].apply(extract_main_domain)

In [ ]:
# check one of the dataframes
df_NL.head()

In [ ]:
# review the list of unique domains as they appear in the df
df_NL.domain.unique()

In [ ]:
# Combine all pages from one domain into a list of pages
# copy one url from the list of unique domains above
website = 'twickel.nl'
document = df_NL[df_NL['domain'] == website]['text'].tolist() # df_DE should match with df_DE above, or change both

# remove new line characters, count words&characters in each line of the text
clean_document = []
word_count = 0
character_count = 0
for i in document:
    j = i.replace('\n', ' ').replace('\r', '')
    w = len(j.split())
    word_count += w
    character_count +=len(j)
    clean_document.append(j)

print('word_count:', word_count, '\ncharacter_count:', character_count)
clean_document

# 1. Compare term frequencies across corpora

## Terms from YOUR LIST across countries

In [ ]:
# list terms to compare
culture = ['tentoonstelling','museum','tuin','park','collectie',
           'exhibition','museum','garden','park','collection',
           'ausstellung','museum','garten','park','sammlung',
           'exposition','musée','jardin','parc','collection']

# PLACE HERE THE LIST YOU PREPARED DURING MONDAY, 23.11 CLASS
YOURLIST = ['']

Create texts as 'documents' to count terms and perform tf-idf

In [ ]:
list_to_count = culture # replace with the name of a list of terms (above)
list_name = 'culture' # replace with the name of a list of terms as a 'string'
dataframes = [df_NL, df_UK, df_DE, df_FR]
whole_word = True   # True: match whole words only; False: match any substring
number_of_terms = len(list_to_count) // 4

# List to gather outputs for all countries in memory
all_country_outputs = []

for i, (cc, df) in enumerate(zip(cc_list, dataframes)):
    terms = list_to_count[i * number_of_terms:(i + 1) * number_of_terms]
    texts = df['text'].fillna('').astype(str).str.lower().tolist()
    urls = df['url'].tolist()

    rows = []
    for term in terms:
        t = re.escape(term.lower())
        pattern = re.compile(r'(?<!Pw)' + t + r'(?!Pw)' if whole_word else t)
        for url, text in zip(urls, texts):
            hits = len(pattern.findall(text))
            if hits:
                rows.append((cc, term, url, hits))   # one row per term and page

    country_out = pd.DataFrame(rows, columns=['cc','term', 'url', 'count'])
    country_out.to_csv(f'{gdrive_path}outputs/{cc}_2024_{list_name}.csv', index=False)
    all_country_outputs.append(country_out)
    print(cc, len(country_out), 'rows saved')

# Combine all countries into a single final 'out' DataFrame in memory
out = pd.concat(all_country_outputs, ignore_index=True)

In [ ]:
out

In [ ]:
import matplotlib.pyplot as plt
number_of_terms = len(list_to_count) // 4 # Calculate subset size per country

country_terms = {} # map each country to its specific subset of terms
for i, country in enumerate(cc_list):
    country_terms[country] = list_to_count[i * number_of_terms : (i + 1) * number_of_terms]

grouped_counts = out.groupby(['cc', 'term'])['count'].sum().to_dict() # group by country and term to sum occurrences

# Create the stacked bar chart
width = 0.45  # Width of the bars

fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(cc_list))
colors = ['#92E0E0', 'red', '#C1E000', '#C192E0', '#E05543', '#E0C200']

for term_index in range(number_of_terms): # loop through list subsets of words in 1 language
    layer_counts = []
    for country in cc_list:
        # Get the specific term for this country at the current layer index
        term_name = country_terms[country][term_index]

        # Search for this term's sum count in grouped_counts
        matching_count = 0
        for (c_code, t_name), val in grouped_counts.items():
            if c_code == country and t_name.lower() == term_name.lower():
                matching_count = val
                break
        layer_counts.append(matching_count)

    # Generate a unified label for the legend combining equivalent translated terms
    legend_label = " / ".join([country_terms[c][term_index] for c in cc_list])

    ax.bar(cc_list, layer_counts, width, label=legend_label, bottom=bottom,
           color=colors[term_index % len(colors)])
    bottom += np.array(layer_counts)

for value in ax.get_yticks():
    ax.axhline(y=value, color='grey', linestyle='--', linewidth=0.5, zorder=0)

ax.set_xlabel('Countries')
ax.set_ylabel('Term Counts')
ax.set_title('Terms related to ' + list_name + ' per country')
ax.tick_params(axis='x')
ax.tick_params(axis='y')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(f'{gdrive_path}outputs/{cc}_2024_{list_name}.png', transparent=True)
plt.show()

## Terms from YOUR LIST related to social inequality, per country

In [ ]:
# REPLACE 'APPLE' and 'PEER' WITH A LIST OF WORDS RELATED TO
# SOCIAL INEQUALITY, PREPARED DURING THE CLASS (23.11)
inequality_terms = {
    'NL': ['appel', 'peer'],
    'UK': ['apple', 'pear'],
    'DE': ['apfel', 'birne'],
    'FR': ['pomme', 'poire'],
}

In [ ]:
# count all terms for all dfs
for cc, df in zip(cc_list, dataframes):
    terms = inequality_terms[cc]
    for term in terms:
        df[term] = df['text'].apply(lambda x: x.lower().count(term) if isinstance(x, str) else 0)

    # Filter for rows where 'kasteel' appears at least once
    mask = (df[terms] > 0).any(axis=1)

    # Filter the DataFrame based on the boolean mask.
    df_filtered = df[mask]

    # Create the five-column table
    filter_list = ['domain','url'] + terms
    result_df = df_filtered[filter_list]
    result_df.to_csv(f'{gdrive_path}outputs/csv/{cc}_2024_inequality.csv', index=False)
    print(cc, len(result_df), 'rows saved')

In [ ]:
# Calculate the sum of values for each specified column
count_pages = False   # False: total occurrences; True: number of pages containing the term

for cc, df in zip(cc_list, dataframes):
    terms = inequality_terms[cc]
    values = (df[terms] > 0).sum() if count_pages else df[terms].sum()

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(terms, values[terms].values, color='red')
    for value in ax.get_yticks():
        ax.axhline(y=value, color='grey', linestyle='--', linewidth=0.5, zorder=0)

    ax.set_title(f'{cc}: inequality terms, 2024')
    ax.set_ylabel('Pages containing the term' if count_pages else 'Total occurrences')
    ax.tick_params(axis='x', rotation=45)
    plt.setp(ax.get_xticklabels(), ha='right')
    for i, v in enumerate(values[terms].values):
        ax.text(i, v, f'{int(v)}', ha='center', va='bottom')   # value label on top of each bar
    ax.set_ylabel('Sum of term frequencies')
    fig.tight_layout()
    fig.savefig(f'{gdrive_path}outputs/{cc}_2024_inequality.png', dpi=200, transparent=True)
    plt.show()


In [ ]:
# generate lists of columns to drop based on terms counted
count_cols = list(dict.fromkeys([t for terms in inequality_terms.values() for t in terms]))       # unique, original order
count_cols = [c for c in count_cols if c not in ('domain', 'url', 'text')]

for cc, df in zip(cc_list, dataframes):
    df.drop(columns=count_cols, errors='ignore', inplace=True)
    print(cc, list(df.columns))
    print(df.head())

## Terms related to nobility according to their gender

In [ ]:
nobility_terms = {
    'NL': [('ridder', 0), ('jonkvrouw', 1),
           ],      # 0 = male titles, 1 = female titles
    'UK': [('king', 0), ('queen', 1)
            ],
    'DE': [('herzog', 0), ('herzogin', 1)
            ],
    'FR': [('duc', 0), ('duchesse', 1)
          ],
}

In [ ]:
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch

# count all terms for all dfs
colors = {0: '#e877f0', 1: '#9f01aa'}
labels = {0: 'Male titles', 1: 'Female titles'}

for cc, df in zip(cc_list, dataframes):
    terms = [term for term, gender in nobility_terms[cc]]
    genders = [gender for term, gender in nobility_terms[cc]]

    # count each term in every page
    for term in terms:
        df[term] = df['text'].apply(lambda x: x.lower().count(term) if isinstance(x, str) else 0)

    values = df[terms].sum()

    # one bar per term, coloured by gender
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(terms, values[terms].values, color=[colors[g] for g in genders])
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color='grey', linestyle='--', linewidth=0.5)
    ax.set_title(f'{cc}: nobility titles, 2024')
    ax.set_ylabel('Total occurrences')
    ax.tick_params(axis='x', rotation=45)
    plt.setp(ax.get_xticklabels(), ha='right')
    ax.legend(handles=[Patch(color=colors[g], label=labels[g]) for g in (0, 1)])
    for i, v in enumerate(values[terms].values):
        ax.text(i, v, f'{int(v)}', ha='center', va='bottom')
    fig.tight_layout()
    fig.savefig(f'{gdrive_path}outputs/{cc}_2024_nobility.png', dpi=200)
    plt.show()

In [ ]:
# generate lists of columns to drop based on terms counted
count_cols = list(dict.fromkeys([t for pairs in nobility_terms.values() for t, gender in pairs]))
count_cols = [c for c in count_cols if c not in ('domain', 'url', 'text')]

for cc, df in zip(cc_list, dataframes):
    df.drop(columns=count_cols, errors='ignore', inplace=True)
    print(cc, list(df.columns))
    #print(df.head())

# 2. Collocation analysis

## Tokenize and process the text

Install _spacy_ module to turn raw text into structured linguistic data like tokens (word-equivalents) or grammatical roles (nouns, adjectives)

https://spacy.io/usage/spacy-101

! DO NOT RESTART AFTER EXECUTING THE NEXT CELL !

In [ ]:
%pip install -q spacy
!python -m spacy download en_core_web_sm
!python -m spacy download nl_core_news_sm
!python -m spacy download de_core_news_sm
!python -m spacy download fr_core_news_sm

What is a token? Learn more in this lecture (optional)

https://media.ccc.de/v/39c3-51-ways-to-spell-the-image-giraffe-the-hidden-politics-of-token-languages-in-generative-ai

Use spaCy to lemmatize the corpus (e.g.
- was, were, is, are → be
- ate, eaten, eats → eat
- swims, swimming, swum → swim
)

and to filter stopwords from the collocates.

Check a window of _five_ words on either side of a target word.

In [ ]:
import spacy
from collections import Counter

In [ ]:
# fundction to extract 'context' words
def extract_contexts(df, nlp, target_lemmas, window=5):
    target_lemmas = {term.lower() for term in target_lemmas}
    context_rows = []
    corpus_counts = Counter()
    target_occurrences = 0

    for url, text, domain in df.itertuples(index=False, name=None):
        if not isinstance(text, str):
            continue
        doc = nlp(text)
        lemmas = [token.lemma_.lower() for token in doc]
        for token, lemma in zip(doc, lemmas):
            if token.is_alpha and not token.is_stop and lemma not in target_lemmas:
                corpus_counts[lemma] += 1
        for i, lemma in enumerate(lemmas):
            if lemma not in target_lemmas:
                continue
            target_occurrences += 1
            start = max(0, i - window)
            end = min(len(doc), i + window + 1)
            for j in range(start, end):
                if j == i:
                    continue
                token = doc[j]
                collocate = lemmas[j]
                if (
                    token.is_alpha
                    and not token.is_stop
                    and collocate not in target_lemmas
                ):
                    context_rows.append(
                        {
                            "collocate": collocate,
                            "domain": domain,
                            "url": url,
                        }
                    )

    contexts = pd.DataFrame(
        context_rows,
        columns=["collocate", "domain", "url"],
    )

    corpus_frequencies = pd.DataFrame(
        corpus_counts.items(),
        columns=["collocate", "corpus_frequency"],
    )

    return contexts, corpus_frequencies, target_occurrences

In [ ]:
df = df_UK # choose one country
nlp = spacy.load("en_core_web_sm") # load the corresponding token classification model
target_lemmas = {"worker", "servant"} # WORDS TO COUNT

In [ ]:
contexts, corpus_frequencies, target_occurrences = extract_contexts(
    df[['url', 'text', 'domain']],
    nlp,
    target_lemmas,
    window=5,
)

print(f"Target occurrences: {target_occurrences}")
print(f"Context-word occurrences: {len(contexts)}")

display(contexts.head(10))
display(corpus_frequencies.sort_values("corpus_frequency", ascending=False).head(10))

In [ ]:
contexts

Now rank the collocations by reviewing each word’s number of co-occurrences, the number of distinct domains it appears in, and its context_lift. A higher lift means the word is more common near the target than in the corpus overall

In [ ]:
def rank_collocations(
    contexts,
    corpus_frequencies,
    target_occurrences,
    min_count,
    min_domains,
):
    columns = [
        "collocate",
        "cooccurrences",
        "domains",
        "context_lift",
        "target_occurrences",
    ]

    if contexts.empty or corpus_frequencies.empty:
        return pd.DataFrame(columns=columns)

    # Count co-occurrences of each collocate
    counts = contexts.groupby("collocate").size().rename("cooccurrences")

    # Count unique domains each collocate appears in
    domain_counts = (
        contexts.groupby("collocate")["domain"]
        .nunique()
        .rename("domains")
    )

    # Merge results with corpus frequencies
    results = pd.concat([counts, domain_counts], axis=1).reset_index()
    results = results.merge(corpus_frequencies, on="collocate")

    context_total = len(contexts)
    corpus_total = corpus_frequencies["corpus_frequency"].sum()

    # Calculate context lift
    results["context_lift"] = (
        (results["cooccurrences"] / context_total)
        / (results["corpus_frequency"] / corpus_total)
    )

    results["target_occurrences"] = target_occurrences

    # Apply thresholds
    results = results[
        (results["cooccurrences"] >= min_count)
        & (results["domains"] >= min_domains)
    ]

    if results.empty:
        return pd.DataFrame(columns=columns)

    # Sort by context lift and co-occurrences
    return results[columns].sort_values(
        ["context_lift", "cooccurrences"],
        ascending=[False, False],
    ).reset_index(drop=True)

In [ ]:
collocations = rank_collocations(
    contexts,
    corpus_frequencies,
    target_occurrences,
    min_count=2, # change
    min_domains=1,
)

display(collocations.head(30))

In [ ]:
collocations.cooccurrences.max(10)

Change the corpus (country)

In [ ]:
df = df_UK # change to a different corpus, for example UK
nlp = spacy.load('en_core_web_sm') # load the corresponding token model
target_lemmas = {''} # WRITE YOUR WORDS HERE

In [ ]:
# call relevant functions here to get the collocations and then to rank them
# you should copy the content of some of the cells above

# 3. Corpus modeling with word2vec

- What is Word2Vec: https://en.wikipedia.org/wiki/Word2Vec

In [ ]:
!pip install gensim

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
import gensim
from nltk.tokenize import word_tokenize

df=df_FR #### CHANGE df0 TO THE DATAFRAME YOU ANALYSE

# X is a list of tokenized texts (i.e. list of lists of tokens)
X = [word_tokenize(item) for item in df.text.tolist()]
#print(X[0:3])
model = gensim.models.Word2Vec(X, min_count=6, vector_size=200) # min_count: how many times a word appears in the corpus; size: number of dimensions

Observe keywords that may be characteristic in the corpus on heritage homes, such as 'castle', 'garden', 'party', 'princess'; try also words related to less obvious themes, like 'servant'

You can ask for 'negative' or 'positive' similarity, and explore how these bring up terms that are opposite to the meaning in a variety of ways.


In [ ]:
model.wv.most_similar(positive=["chateau"], topn=20)

In [ ]:
model.wv.most_similar(negative=["prince"], topn=12)

In [ ]:
model.wv.most_similar(negative=["roi"], topn=12)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# 1. Use the existing trained model and target word
target_word = "prince"
similar_words = model.wv.most_similar(positive=[target_word], topn=30) # change this

# 2. Build the Network Graph
G = nx.Graph()
G.add_node(target_word)

# Store similarities to scale font sizes later
similarities = {target_word: 1.0}
for word, similarity in similar_words:
    G.add_node(word)
    G.add_edge(target_word, word, weight=similarity)
    similarities[word] = similarity

# 3. Configure layout and visualize the graph
plt.figure(figsize=(14, 12))
pos = nx.spring_layout(G, k=0.18, iterations=50, seed=42)

# Draw only the central target node with a circle background for clear emphasis
nx.draw_networkx_nodes(
    G, pos,
    nodelist=[target_word],
    node_size=3000,
    node_color='#FF5533',
    alpha=0.9
)

# Draw edges
nx.draw_networkx_edges(G, pos, width=1.2, alpha=0.3, edge_color='gray')

# Draw labels: central word gets a fixed large font, other words scale based on similarity
for node in G.nodes():
    if node == target_word:
        font_size = 14
        font_color = 'white'
        font_weight = 'bold'
    else:
        # Scale font size between 8 and 16 based on similarity score
        score = similarities[node]
        font_size = 6 + (score * 40)
        font_color = 'black'
        font_weight = 'normal'

    nx.draw_networkx_labels(
        G, pos,
        labels={node: node},
        font_size=font_size,
        font_color=font_color,
        font_weight=font_weight,
        font_family='sans-serif'
    )

plt.title(f"Word Network: 30 Closest Words scaled by Similarity to '{target_word}' (FR Corpus)", fontsize=16, fontweight='bold', pad=20)
plt.axis('off')
plt.tight_layout()
plt.show()

# 4. Topic Modelling

In [ ]:
import sklearn.feature_extraction.text as text

df=df_UK # CHANGE df0 TO THE DATAFRAME YOU ARE ANALYSING

# min_df: ignore words occurring in fewer than `n` documents
# stop_words: ignore very common words ("the", "and", "or", "to", ...)
vec = text.CountVectorizer(lowercase=True, min_df=100)
dtm = vec.fit_transform(df['text'])

In [ ]:
print(f'Shape of document-term matrix: {dtm.shape}. '
      f'Number of tokens {dtm.sum()}')

In [ ]:
import sklearn.decomposition as decomposition
NUM_TOPICS = 10
lda_model = decomposition.LatentDirichletAllocation(
    n_components=NUM_TOPICS, learning_method='online', random_state=1)
lda_Z = lda_model.fit_transform(dtm)
print(lda_Z.shape)  # (NO_DOCUMENTS, NO_TOPICS)

document_topic_distributions = lda_model.fit_transform(dtm)

top_n=10
for idx, topic in enumerate(lda_model.components_):
  print("Topic %d:" % (idx))
  print([(vec.get_feature_names_out()[i], topic[i])
  for i in topic.argsort()[:-top_n - 1:-1]])

In [ ]:
no_top_words = 12
no_top_documents = 5
lda_H = lda_model.components_
tf_feature_names = vec.get_feature_names_out()

df=df_NL # CHANGE df0 TO THE DATAFRAME YOU ARE ANALYSING

def display_topics(H, Z, feature_names, docs, no_top_words, no_top_documents):
    for idx, topic in enumerate(H):
        print("Topic %d:" % (idx))
        print("KEYWORDS", " ".join([feature_names[i]
                        for i in topic.argsort()[:-no_top_words - 1:-1]]))
        top_doc_indices = np.argsort( Z[:,
                                        idx] )[::-1][0:no_top_documents]
        # good for checking which documents are the most characteristic for certain topics
        for doc_index in top_doc_indices:
            print("TOP DOCS", docs[doc_index])

display_topics(lda_H, lda_Z, tf_feature_names, df['url'].tolist(), no_top_words, no_top_documents)